In [ ]:
import urllib.request
import pandas as pd

df = pd.read_csv("Dataset/dataset.csv")
print(f"Dataset loaded! Shape: {df.shape}")

In [ ]:
!pip install opencv-python mediapipe

# **1. Dataset**

## Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Dataset Loading

In [ ]:
DATA_PATH = "Dataset/dataset.csv"
df = pd.read_csv(DATA_PATH)

## Landmark Selection

In [ ]:
selected_columns = [
    "subject",
    "upperbody_label",
    "nose_x", "nose_y", "nose_z",
    "left_eye_x", "left_eye_y", "left_eye_z",
    "right_eye_x", "right_eye_y", "right_eye_z",
    "left_ear_x", "left_ear_y", "left_ear_z",
    "right_ear_x", "right_ear_y", "right_ear_z",
    "left_shoulder_x", "left_shoulder_y", "left_shoulder_z",
    "right_shoulder_x", "right_shoulder_y", "right_shoulder_z",
    "left_hip_x", "left_hip_y", "left_hip_z",
    "right_hip_x", "right_hip_y", "right_hip_z"
]

missing_cols = [col for col in selected_columns if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in dataset: {missing_cols}")

df_posture = df[selected_columns].copy()

print("Selected dataset shape:", df_posture.shape)
print("Total missing values:", df_posture.isnull().sum().sum())

## Midpoint Construction

In [ ]:
# Thresholds for future consistency with the real-time pipeline
HEAD_Z_CONF_THR = 0.6
SHOULDER_CONF_THR = 0.6
HIP_CONF_THR = 0.6


# Compute the midpoint between two bilateral landmarks
def add_midpoint(df, left_prefix, right_prefix, out_prefix):
    # Check whether all coordinates are available for the left landmark
    left_available = df[
        [f"{left_prefix}_{axis}" for axis in ["x", "y", "z"]]
    ].notna().all(axis=1)

    # Check whether all coordinates are available for the right landmark
    right_available = df[
        [f"{right_prefix}_{axis}" for axis in ["x", "y", "z"]]
    ].notna().all(axis=1)

    # The midpoint is available only when both landmarks are present
    available = left_available & right_available

    # Compute midpoint coordinates for each axis
    for axis in ["x", "y", "z"]:
        midpoint = (df[f"{left_prefix}_{axis}"] + df[f"{right_prefix}_{axis}"]) / 2
        df[f"{out_prefix}_{axis}"] = np.where(available, midpoint, np.nan)

    # Store binary availability information
    df[f"{out_prefix}_available"] = available

    # Store binary confidence information
    # The offline dataset does not provide continuous visibility scores
    df[f"{out_prefix}_conf"] = available.astype(float)

    return df


# Compute bilateral midpoints used for posture representation
df_posture = add_midpoint(df_posture, "left_shoulder", "right_shoulder", "mid_shoulder")
df_posture = add_midpoint(df_posture, "left_hip", "right_hip", "mid_hip")
df_posture = add_midpoint(df_posture, "left_ear", "right_ear", "mid_ear")
df_posture = add_midpoint(df_posture, "left_eye", "right_eye", "mid_eye")


# Define reliability flags for the main structural midpoints
df_posture["mid_shoulder_reliable"] = df_posture["mid_shoulder_available"]
df_posture["mid_hip_reliable"] = df_posture["mid_hip_available"]


# Check availability of candidate landmarks for head center construction
eye_available = df_posture[
    [
        "left_eye_x", "left_eye_y", "left_eye_z",
        "right_eye_x", "right_eye_y", "right_eye_z"
    ]
].notna().all(axis=1)

ear_available = df_posture[
    [
        "left_ear_x", "left_ear_y", "left_ear_z",
        "right_ear_x", "right_ear_y", "right_ear_z"
    ]
].notna().all(axis=1)

nose_available = df_posture[
    ["nose_x", "nose_y", "nose_z"]
].notna().all(axis=1)


# Construct head center coordinates using a hierarchical fallback strategy
# Priority: mid-eye -> mid-ear -> nose
for axis in ["x", "y", "z"]:
    df_posture[f"head_center_{axis}"] = np.where(
        eye_available,
        df_posture[f"mid_eye_{axis}"],
        np.where(
            ear_available,
            df_posture[f"mid_ear_{axis}"],
            np.where(
                nose_available,
                df_posture[f"nose_{axis}"],
                np.nan
            )
        )
    )


df_posture["head_center_source"] = np.where(
    eye_available,
    "mid_eye",
    np.where(
        ear_available,
        "mid_ear",
        np.where(nose_available, "nose", "missing")
    )
)


# Store availability and binary confidence for head center
df_posture["head_center_available"] = (
    eye_available |
    ear_available |
    nose_available
)

df_posture["head_center_conf"] = df_posture["head_center_available"].astype(float)
df_posture["head_center_z_reliable"] = df_posture["head_center_source"].isin(
    ["mid_eye", "mid_ear"]
)
df_posture["upper_z_reliable"] = (
    df_posture["head_center_z_reliable"] &
    df_posture["mid_shoulder_reliable"]
)


# Quick inspection of the generated midpoint and reliability features
print(
    df_posture[
        [
            "mid_shoulder_x", "mid_shoulder_y", "mid_shoulder_z",
            "mid_shoulder_available", "mid_shoulder_conf", "mid_shoulder_reliable",

            "mid_hip_x", "mid_hip_y", "mid_hip_z",
            "mid_hip_available", "mid_hip_conf", "mid_hip_reliable",

            "mid_ear_x", "mid_ear_y", "mid_ear_z",
            "mid_ear_available", "mid_ear_conf",

            "mid_eye_x", "mid_eye_y", "mid_eye_z",
            "mid_eye_available", "mid_eye_conf",

            "head_center_x", "head_center_y", "head_center_z",
            "head_center_source", "head_center_available", "head_center_conf",

            "head_center_z_reliable",
            "upper_z_reliable"
        ]
    ].head()
)

## Feature Definition

### Primary Features

In [ ]:
eps = 1e-6

# Wrap angle to [-90, 90)
def wrap_angle_90(angle_deg):
    return ((angle_deg + 90) % 180) - 90


# Signed angular difference in [-180, 180)
def angle_diff_deg(a, b):
    return (a - b + 180) % 360 - 180


# Required columns check
required_columns = [
    "left_shoulder_x", "left_shoulder_y",
    "right_shoulder_x", "right_shoulder_y",
    "left_eye_x", "left_eye_y", "right_eye_x", "right_eye_y",
    "left_ear_x", "left_ear_y", "right_ear_x", "right_ear_y",
    "head_center_x", "head_center_y", "head_center_z",
    "mid_shoulder_x", "mid_shoulder_y", "mid_shoulder_z",
    "upper_z_reliable"
]

missing_columns = [col for col in required_columns if col not in df_posture.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")


# =========================
# 1) SHOULDERS GEOMETRY
# =========================
dx_shoulder = df_posture["right_shoulder_x"] - df_posture["left_shoulder_x"]
dy_shoulder = df_posture["right_shoulder_y"] - df_posture["left_shoulder_y"]

df_posture["shoulder_slope"] = wrap_angle_90(
    np.degrees(np.arctan2(dy_shoulder, dx_shoulder))
)

df_posture["shoulder_width"] = np.hypot(dx_shoulder, dy_shoulder)
df_posture["shoulder_width"] = df_posture["shoulder_width"].replace(0, np.nan)
df_posture["shoulder_width_safe"] = df_posture["shoulder_width"].fillna(eps)

u_x = dx_shoulder / df_posture["shoulder_width_safe"]
u_y = dy_shoulder / df_posture["shoulder_width_safe"]

n_x = -u_y
n_y = u_x

flip_mask = n_y > 0
n_x = np.where(flip_mask, -n_x, n_x)
n_y = np.where(flip_mask, -n_y, n_y)


# =========================
# 2) HEAD ORIENTATION
# =========================
eye_available = df_posture[
    ["left_eye_x", "left_eye_y", "right_eye_x", "right_eye_y"]
].notna().all(axis=1)

ear_available = df_posture[
    ["left_ear_x", "left_ear_y", "right_ear_x", "right_ear_y"]
].notna().all(axis=1)

dx_eye = df_posture["right_eye_x"] - df_posture["left_eye_x"]
dy_eye = df_posture["right_eye_y"] - df_posture["left_eye_y"]
head_tilt_eye = wrap_angle_90(np.degrees(np.arctan2(dy_eye, dx_eye)))

dx_ear = df_posture["right_ear_x"] - df_posture["left_ear_x"]
dy_ear = df_posture["right_ear_y"] - df_posture["left_ear_y"]
head_tilt_ear = wrap_angle_90(np.degrees(np.arctan2(dy_ear, dx_ear)))

df_posture["head_tilt"] = np.where(
    eye_available,
    head_tilt_eye,
    np.where(ear_available, head_tilt_ear, np.nan)
)


# =========================
# 3) HEAD RELATIVE TO SHOULDERS
# =========================
H_x = df_posture["head_center_x"] - df_posture["mid_shoulder_x"]
H_y = df_posture["head_center_y"] - df_posture["mid_shoulder_y"]
H_z = df_posture["head_center_z"] - df_posture["mid_shoulder_z"]

proj_shoulder_axis   = H_x * u_x + H_y * u_y
proj_shoulder_normal = H_x * n_x + H_y * n_y

df_posture["head_lateral_ratio"] = proj_shoulder_axis / df_posture["shoulder_width_safe"]
df_posture["head_y_ratio"]       = proj_shoulder_normal / df_posture["shoulder_width_safe"]


# =========================
# 4) 2D POSTURAL ANGLES
# =========================
df_posture["upper_body_inclination"] = wrap_angle_90(
    np.degrees(np.arctan2(proj_shoulder_axis, proj_shoulder_normal + eps))
)

norm_H_2d = np.sqrt(H_x**2 + H_y**2) + eps
cos_theta = (-H_y) / norm_H_2d
cos_theta = np.clip(cos_theta, -1.0, 1.0)

df_posture["head_neck_vertical_angle"] = np.degrees(np.arccos(cos_theta))


# =========================
# 5) RELATIVE ANGULAR FEATURES
# =========================
df_posture["head_shoulder_alignment"] = angle_diff_deg(
    df_posture["head_tilt"],
    df_posture["shoulder_slope"]
)

df_posture["head_trunk_diff"] = angle_diff_deg(
    df_posture["head_tilt"],
    df_posture["upper_body_inclination"]
)

# =========================
# 6) DIAGNOSTIC (non entra nel modello)
# =========================
df_posture["head_depth_ratio_diag"] = np.where(
    df_posture["upper_z_reliable"],
    H_z / df_posture["shoulder_width_safe"],
    np.nan
)


# =========================
# 7) FINAL PRIMARY FEATURE LIST — 8 feature
# =========================
primary_features = [
    "head_tilt",
    "shoulder_slope",
    "head_lateral_ratio",
    "head_y_ratio",
    "head_neck_vertical_angle",
    "head_shoulder_alignment",
    "upper_body_inclination",
    "head_trunk_diff",
]

missing_primary_features = [col for col in primary_features if col not in df_posture.columns]
if missing_primary_features:
    raise KeyError(f"Missing primary features: {missing_primary_features}")

print("PRIMARY FEATURES:")
print(primary_features)

### Support Features

The support features extend the postural representation by incorporating additional information derived from the pelvis, whenever these landmarks are available and sufficiently reliable. These features do not constitute the core of the dataset; rather, they provide an additional level of postural description that is particularly useful for modeling trunk alignment.

More specifically, this group includes measures describing the geometry of the segment between the hips and the shoulders, such as trunk lateral inclination, forward/backward trunk inclination, and its depth-related component. In addition, an anatomical feature is introduced to approximate the relationship between the neck/head direction and the trunk direction.

Since these measures depend on the correct observation of the hip landmarks, they are computed only when the required anatomical references are available. In this way, the support features maintain a complementary role: they enrich the dataset when reliable information is present, without compromising the overall robustness of the pipeline.

In [ ]:
# =========================
# VETTORI
# =========================
T_x = df_posture["mid_shoulder_x"] - df_posture["mid_hip_x"]
T_y = df_posture["mid_shoulder_y"] - df_posture["mid_hip_y"]
T_z = df_posture["mid_shoulder_z"] - df_posture["mid_hip_z"]

# =========================
# RELIABILITY FLAG
# =========================
df_posture["trunk_support_reliable"] = (
    df_posture["mid_shoulder_reliable"] &
    df_posture["mid_hip_reliable"]
)

# =========================
# trunk_forward_backward_angle — FIX: -T_y
# =========================
trunk_fba_raw = np.degrees(
    np.arctan2(T_z, -T_y + 1e-6)
)

df_posture["trunk_forward_backward_angle"] = np.where(
    df_posture["trunk_support_reliable"],
    trunk_fba_raw,
    np.nan
)

# =========================
# EXTRA FEATURES — normalizzate per shoulder_width
# Migliorano la discriminazione TLL/TLR su soggetti mai visti
# =========================

# Asimmetria z delle spalle (rotazione del busto sul piano trasversale)
df_posture["shoulder_z_asym"] = (
    (df_posture["left_shoulder_z"] - df_posture["right_shoulder_z"])
    / df_posture["shoulder_width_safe"]
)

# Offset laterale bacino rispetto alle spalle (normalizzato)
df_posture["hip_shoulder_lateral"] = (
    (df_posture["mid_hip_x"] - df_posture["mid_shoulder_x"]) * u_x
    + (df_posture["mid_hip_y"] - df_posture["mid_shoulder_y"]) * u_y
) / df_posture["shoulder_width_safe"]

# Asimmetria altezza orecchie
df_posture["ear_height_asym"] = (
    (df_posture["left_ear_y"] - df_posture["right_ear_y"])
    / df_posture["shoulder_width_safe"]
)

extra_features = ["shoulder_z_asym", "hip_shoulder_lateral", "ear_height_asym"]

# =========================
# FINAL SUPPORT FEATURE LIST — 1 sagittale + 3 extra
# =========================
support_features = ["trunk_forward_backward_angle"] + extra_features

missing_support = [col for col in support_features if col not in df_posture.columns]
if missing_support:
    raise KeyError(f"Missing support features: {missing_support}")

print("SUPPORT FEATURES:")
print(support_features)

print("\nQuick check trunk_forward_backward_angle:")
print(df_posture["trunk_forward_backward_angle"].describe().round(3))
print(f"\nValori NaN: {df_posture['trunk_forward_backward_angle'].isna().sum()}")
print(f"Valori fuori [-30, 30]: {((df_posture['trunk_forward_backward_angle'].abs() > 30) & df_posture['trunk_forward_backward_angle'].notna()).sum()}")

print("\nQuick check extra features:")
for feat in extra_features:
    s = df_posture[feat]
    print(f"  {feat}: mean={s.mean():.3f}, std={s.std():.3f}, NaN={s.isna().sum()}")

## Final Dataset Construction

At this stage, the final dataset containing the features derived from the selected and transformed landmarks is assembled.

The features used by the model are obtained by combining the **primary features**, based on head and shoulder information, with the **support features**, derived from trunk-related geometry when the required landmarks are available. This integration makes it possible to preserve a balance between robustness and completeness in the postural representation.

During the previous stages, several intermediate variables are also generated, including vector components, normalized quantities, and diagnostic features. These variables are retained in the working DataFrame to ensure traceability and to facilitate debugging. However, they are not included in the final model dataset, in order to avoid redundancy and to preserve a compact and interpretable feature representation.

Two distinct datasets are generated:

- **Model dataset**: contains only the features used as model input, together with the subject identifier and the postural class label. This dataset represents the main output of the feature engineering pipeline.

- **Debug dataset**: includes, in addition to the model features, diagnostic variables and reliability flags. This version is intended to support transformation verification, feature inspection, and possible downstream error analysis.

Both datasets are saved in `.csv` format so that they can be reused in the subsequent stages of the project.

Finally, a set of basic structural checks is performed, including verification of dataset dimensionality, number of extracted features, and the overall amount of missing values. These checks ensure the correctness of the dataset construction process without entering the exploratory analysis stage, which will be addressed separately.

In [ ]:
# =========================
# FINAL FEATURE LIST — 8 primary + 4 support (1 sagittale + 3 extra)
# =========================
all_feature_columns = primary_features + support_features

diagnostic_columns = [
    "shoulder_width",
    "shoulder_width_safe",
    "head_depth_ratio_diag",
]

id_columns = ["subject", "upperbody_label"]

control_columns = [
    "head_center_source",
    "head_center_z_reliable",
    "upper_z_reliable",
    "mid_shoulder_reliable",
    "trunk_support_reliable",
]

required_columns = (
    id_columns
    + all_feature_columns
    + diagnostic_columns
    + control_columns
)

missing_columns = [col for col in required_columns if col not in df_posture.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

# =========================
# FINAL MODEL DATASET
# =========================
df_features = df_posture[id_columns + all_feature_columns].copy()

df_features_debug = df_posture[
    id_columns + all_feature_columns + diagnostic_columns + control_columns
].copy()

if df_features.empty:
    raise ValueError("The final feature dataset is empty.")

print("Model dataset missing values:")
print(df_features.isnull().sum().sort_values(ascending=False))

print("\nDatasets saved.")
print("Model dataset shape:", df_features.shape)
print("Feature columns:", all_feature_columns)

df_features.to_csv("posture_features.csv", index=False)
df_features_debug.to_csv("posture_features_debug.csv", index=False)

# **2. Utility Functions**

## Costants and features

In [ ]:
EPS = 1e-6

HEAD_Z_CONF_THR   = 0.6
SHOULDER_CONF_THR = 0.6
HIP_CONF_THR      = 0.6
EAR_Z_CONF_THR    = 0.7

FEATURE_NAMES = [
    # PRIMARY
    "head_tilt",
    "shoulder_slope",
    "head_lateral_ratio",
    "head_y_ratio",
    "head_neck_vertical_angle",
    "head_shoulder_alignment",
    "upper_body_inclination",
    "head_trunk_diff",
    # SUPPORT
    "trunk_forward_backward_angle",
    # EXTRA — normalizzate per shoulder_width, migliorano TLL/TLR
    "shoulder_z_asym",
    "hip_shoulder_lateral",
    "ear_height_asym",
]

CONTROL_NAMES = [
    "head_center_source",
    "head_center_conf",
    "head_center_z_reliable",
    "mid_shoulder_conf",
    "mid_shoulder_reliable",
    "upper_z_reliable",
    "trunk_support_reliable",
]

## Basic Helpers

In [ ]:
def _as_landmark(p):
    try:
        arr = np.asarray(p, dtype=float)
    except Exception:
        return np.array([np.nan, np.nan, np.nan], dtype=float)
    if arr.shape != (3,):
        return np.array([np.nan, np.nan, np.nan], dtype=float)
    return arr


def _as_visibility(v, default=np.nan):
    try:
        if v is None:
            return float(default)
        v = float(v)
        return v if np.isfinite(v) else float(default)
    except Exception:
        return float(default)


def is_valid_landmark(p):
    p = _as_landmark(p)
    return p.shape == (3,) and np.isfinite(p).all()


def safe_norm(v):
    v = _as_landmark(v)
    if not np.isfinite(v).all():
        return np.nan
    return float(np.linalg.norm(v) + EPS)


def safe_divide(num, den):
    if not np.isfinite(num) or not np.isfinite(den) or abs(den) < EPS:
        return np.nan
    return float(num / den)


def wrap_angle_90(angle_deg):
    return ((angle_deg + 90.0) % 180.0) - 90.0


def angle_diff_deg(a, b):
    if not np.isfinite(a) or not np.isfinite(b):
        return np.nan
    return float((a - b + 180.0) % 360.0 - 180.0)


def combine_visibility(v1, v2):
    v1 = _as_visibility(v1, default=np.nan)
    v2 = _as_visibility(v2, default=np.nan)
    if np.isfinite(v1) and np.isfinite(v2):
        return float(min(v1, v2))
    if np.isfinite(v1):
        return float(v1)
    if np.isfinite(v2):
        return float(v2)
    return np.nan

## Midpoint and HeadCenter

In [ ]:
def compute_midpoint_with_conf(p1, p2, vis1=None, vis2=None):
    p1 = _as_landmark(p1)
    p2 = _as_landmark(p2)
    if not is_valid_landmark(p1) or not is_valid_landmark(p2):
        return np.array([np.nan, np.nan, np.nan]), False, 0.0
    mid  = (p1 + p2) / 2.0
    conf = combine_visibility(vis1, vis2)
    conf = float(conf) if np.isfinite(conf) else 1.0
    return mid, True, conf


def compute_head_center(
    nose, l_eye, r_eye, l_ear, r_ear,
    nose_vis=None, l_eye_vis=None, r_eye_vis=None,
    l_ear_vis=None, r_ear_vis=None,
):
    nose  = _as_landmark(nose)
    l_eye = _as_landmark(l_eye)
    r_eye = _as_landmark(r_eye)
    l_ear = _as_landmark(l_ear)
    r_ear = _as_landmark(r_ear)

    eye_conf = combine_visibility(l_eye_vis, r_eye_vis)
    ear_conf = combine_visibility(l_ear_vis, r_ear_vis)
    nose_conf = _as_visibility(nose_vis, default=0.0)

    EYE_THR  = 0.6
    EAR_THR  = 0.6
    NOSE_THR = 0.5

    if (
        is_valid_landmark(l_eye) and is_valid_landmark(r_eye) and
        np.isfinite(eye_conf) and eye_conf >= EYE_THR
    ):
        mid = (l_eye + r_eye) / 2.0
        return mid, "mid_eye", float(eye_conf), True

    if (
        is_valid_landmark(l_ear) and is_valid_landmark(r_ear) and
        np.isfinite(ear_conf) and ear_conf >= EAR_THR
    ):
        mid = (l_ear + r_ear) / 2.0
        return mid, "mid_ear", float(ear_conf), True

    if is_valid_landmark(nose) and np.isfinite(nose_conf) and nose_conf >= NOSE_THR:
        return nose.copy(), "nose", float(nose_conf), False

    return np.array([np.nan, np.nan, np.nan]), "missing", 0.0, False

## Shoulder geometry

In [ ]:
def compute_shoulder_width(l_shoulder, r_shoulder):
    l = _as_landmark(l_shoulder)
    r = _as_landmark(r_shoulder)
    if not is_valid_landmark(l) or not is_valid_landmark(r):
        return np.nan
    return float(np.linalg.norm(r[:2] - l[:2]) + EPS)


def compute_shoulder_slope(l_shoulder, r_shoulder):
    l = _as_landmark(l_shoulder)
    r = _as_landmark(r_shoulder)
    if not is_valid_landmark(l) or not is_valid_landmark(r):
        return np.nan
    dx = r[0] - l[0]
    dy = r[1] - l[1]
    return float(wrap_angle_90(np.degrees(np.arctan2(dy, dx))))


def compute_shoulder_axes(l_shoulder, r_shoulder):
    l = _as_landmark(l_shoulder)
    r = _as_landmark(r_shoulder)
    if not is_valid_landmark(l) or not is_valid_landmark(r):
        return None, None
    dx   = r[0] - l[0]
    dy   = r[1] - l[1]
    norm = float(np.hypot(dx, dy) + EPS)
    ux, uy = dx / norm, dy / norm
    nx, ny = -uy, ux
    if ny > 0:
        nx, ny = -nx, -ny
    return np.array([ux, uy]), np.array([nx, ny])

## Primary features

In [ ]:
def compute_head_tilt(l_eye, r_eye, l_ear, r_ear):
    l_eye = _as_landmark(l_eye)
    r_eye = _as_landmark(r_eye)
    l_ear = _as_landmark(l_ear)
    r_ear = _as_landmark(r_ear)
    if is_valid_landmark(l_eye) and is_valid_landmark(r_eye):
        return float(wrap_angle_90(np.degrees(
            np.arctan2(r_eye[1] - l_eye[1], r_eye[0] - l_eye[0])
        )))
    if is_valid_landmark(l_ear) and is_valid_landmark(r_ear):
        return float(wrap_angle_90(np.degrees(
            np.arctan2(r_ear[1] - l_ear[1], r_ear[0] - l_ear[0])
        )))
    return np.nan


def compute_head_lateral_ratio(head_center, mid_shoulder, u, shoulder_width):
    head = _as_landmark(head_center)
    mid  = _as_landmark(mid_shoulder)
    if u is None or not is_valid_landmark(head) or not is_valid_landmark(mid):
        return np.nan
    hx = head[0] - mid[0]
    hy = head[1] - mid[1]
    return safe_divide(hx * u[0] + hy * u[1], shoulder_width)


def compute_head_y_ratio(head_center, mid_shoulder, n, shoulder_width):
    head = _as_landmark(head_center)
    mid  = _as_landmark(mid_shoulder)
    if n is None or not is_valid_landmark(head) or not is_valid_landmark(mid):
        return np.nan
    hx = head[0] - mid[0]
    hy = head[1] - mid[1]
    return safe_divide(hx * n[0] + hy * n[1], shoulder_width)


def compute_head_neck_vertical_angle(head_center, mid_shoulder):
    head = _as_landmark(head_center)
    mid  = _as_landmark(mid_shoulder)
    if not is_valid_landmark(head) or not is_valid_landmark(mid):
        return np.nan
    dx    = head[0] - mid[0]
    dy    = head[1] - mid[1]
    cos_t = np.clip((-dy) / (np.sqrt(dx*dx + dy*dy) + EPS), -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_t)))


def compute_head_shoulder_alignment(head_tilt, shoulder_slope):
    diff = angle_diff_deg(head_tilt, shoulder_slope)
    return float(diff) if np.isfinite(diff) else np.nan


def compute_upper_body_inclination(head_center, mid_shoulder, u, n):
    head = _as_landmark(head_center)
    mid  = _as_landmark(mid_shoulder)
    if u is None or n is None or not is_valid_landmark(head) or not is_valid_landmark(mid):
        return np.nan
    hx = head[0] - mid[0]
    hy = head[1] - mid[1]
    return float(wrap_angle_90(np.degrees(
        np.arctan2(hx * u[0] + hy * u[1], hx * n[0] + hy * n[1] + EPS)
    )))


def compute_head_trunk_diff(head_tilt, upper_body_inclination):
    diff = angle_diff_deg(head_tilt, upper_body_inclination)
    return float(diff) if np.isfinite(diff) else np.nan

## Support feature

In [ ]:
def compute_trunk_forward_backward_angle(mid_shoulder, mid_hip,
                                          trunk_support_reliable=True):
    """
    Angolo di inclinazione sagittale del tronco.

    Fix geometrico: in coordinate immagine MediaPipe y cresce verso il basso,
    quindi T_y = shoulder_y - hip_y è negativo (spalle sopra le anche).
    Usando -T_y come denominatore si riporta nel sistema biomeccanico
    (y verso l'alto), producendo angoli ~0° in postura neutra.

    Convenzione output:
      > 0  → forward lean (testa/spalle avanzate rispetto alle anche)
      < 0  → backward lean
      ~0   → postura neutra verticale
    """
    mid_shoulder = _as_landmark(mid_shoulder)
    mid_hip      = _as_landmark(mid_hip)

    if not is_valid_landmark(mid_shoulder) or not is_valid_landmark(mid_hip):
        return np.nan
    if not trunk_support_reliable:
        return np.nan

    tz = mid_shoulder[2] - mid_hip[2]
    ty = mid_shoulder[1] - mid_hip[1]   # negativo in coord. immagine

    return float(np.degrees(np.arctan2(tz, -ty + EPS)))

## Main extraction

In [ ]:
def extract_features(
    nose, l_eye, r_eye, l_ear, r_ear,
    l_shoulder, r_shoulder,
    l_hip=None, r_hip=None,
    nose_vis=None,
    l_eye_vis=None, r_eye_vis=None,
    l_ear_vis=None, r_ear_vis=None,
    l_shoulder_vis=None, r_shoulder_vis=None,
    l_hip_vis=None, r_hip_vis=None,
):
    l_shoulder = _as_landmark(l_shoulder)
    r_shoulder = _as_landmark(r_shoulder)

    mid_shoulder, mid_shoulder_available, mid_shoulder_conf = compute_midpoint_with_conf(
        l_shoulder, r_shoulder, l_shoulder_vis, r_shoulder_vis
    )
    mid_shoulder_reliable = bool(
        mid_shoulder_available and mid_shoulder_conf >= SHOULDER_CONF_THR
    )

    # Anche — opzionali
    l_hip_lm = _as_landmark(l_hip) if l_hip is not None else np.array([np.nan]*3)
    r_hip_lm = _as_landmark(r_hip) if r_hip is not None else np.array([np.nan]*3)
    mid_hip, mid_hip_available, mid_hip_conf = compute_midpoint_with_conf(
        l_hip_lm, r_hip_lm, l_hip_vis, r_hip_vis
    )
    mid_hip_reliable = bool(
        mid_hip_available and mid_hip_conf >= HIP_CONF_THR
    )
    trunk_support_reliable = bool(mid_shoulder_reliable and mid_hip_reliable)

    head_center, _, _, _ = compute_head_center(
        _as_landmark(nose), _as_landmark(l_eye), _as_landmark(r_eye),
        _as_landmark(l_ear), _as_landmark(r_ear),
        nose_vis=nose_vis,
        l_eye_vis=l_eye_vis, r_eye_vis=r_eye_vis,
        l_ear_vis=l_ear_vis, r_ear_vis=r_ear_vis,
    )

    shoulder_width = compute_shoulder_width(l_shoulder, r_shoulder)
    shoulder_slope = compute_shoulder_slope(l_shoulder, r_shoulder)
    head_tilt      = compute_head_tilt(
        _as_landmark(l_eye), _as_landmark(r_eye),
        _as_landmark(l_ear), _as_landmark(r_ear)
    )
    u, n = compute_shoulder_axes(l_shoulder, r_shoulder)

    head_lateral_ratio       = compute_head_lateral_ratio(head_center, mid_shoulder, u, shoulder_width)
    head_y_ratio             = compute_head_y_ratio(head_center, mid_shoulder, n, shoulder_width)
    head_neck_vertical_angle = compute_head_neck_vertical_angle(head_center, mid_shoulder)
    head_shoulder_alignment  = compute_head_shoulder_alignment(head_tilt, shoulder_slope)
    upper_body_inclination   = compute_upper_body_inclination(head_center, mid_shoulder, u, n)
    head_trunk_diff          = compute_head_trunk_diff(head_tilt, upper_body_inclination)

    trunk_fba = compute_trunk_forward_backward_angle(
        mid_shoulder, mid_hip,
        trunk_support_reliable=trunk_support_reliable
    )

    # === EXTRA FEATURES normalizzate per shoulder_width ===
    sw_safe = float(shoulder_width) if (np.isfinite(shoulder_width) and shoulder_width > EPS) else EPS

    if is_valid_landmark(l_shoulder) and is_valid_landmark(r_shoulder):
        shoulder_z_asym = float((l_shoulder[2] - r_shoulder[2]) / sw_safe)
    else:
        shoulder_z_asym = np.nan

    if is_valid_landmark(mid_shoulder) and is_valid_landmark(mid_hip) and u is not None:
        dxh = mid_hip[0] - mid_shoulder[0]
        dyh = mid_hip[1] - mid_shoulder[1]
        hip_shoulder_lateral = float((dxh * u[0] + dyh * u[1]) / sw_safe)
    else:
        hip_shoulder_lateral = np.nan

    l_ear_lm = _as_landmark(l_ear)
    r_ear_lm = _as_landmark(r_ear)
    if is_valid_landmark(l_ear_lm) and is_valid_landmark(r_ear_lm):
        ear_height_asym = float((l_ear_lm[1] - r_ear_lm[1]) / sw_safe)
    else:
        ear_height_asym = np.nan

    return np.array([
        head_tilt,
        shoulder_slope,
        head_lateral_ratio,
        head_y_ratio,
        head_neck_vertical_angle,
        head_shoulder_alignment,
        upper_body_inclination,
        head_trunk_diff,
        trunk_fba,
        shoulder_z_asym,
        hip_shoulder_lateral,
        ear_height_asym,
    ], dtype=float)


def extract_features_as_dict(
    nose, l_eye, r_eye, l_ear, r_ear,
    l_shoulder, r_shoulder,
    l_hip=None, r_hip=None,
    **kwargs
):
    values = extract_features(
        nose, l_eye, r_eye, l_ear, r_ear,
        l_shoulder, r_shoulder,
        l_hip=l_hip, r_hip=r_hip,
        **kwargs
    )
    return dict(zip(FEATURE_NAMES, values))


def extract_features_with_control(
    nose, l_eye, r_eye, l_ear, r_ear,
    l_shoulder, r_shoulder,
    l_hip=None, r_hip=None,
    nose_vis=None,
    l_eye_vis=None, r_eye_vis=None,
    l_ear_vis=None, r_ear_vis=None,
    l_shoulder_vis=None, r_shoulder_vis=None,
    l_hip_vis=None, r_hip_vis=None,
):
    l_shoulder_lm = _as_landmark(l_shoulder)
    r_shoulder_lm = _as_landmark(r_shoulder)
    l_hip_lm      = _as_landmark(l_hip) if l_hip is not None else np.array([np.nan]*3)
    r_hip_lm      = _as_landmark(r_hip) if r_hip is not None else np.array([np.nan]*3)

    mid_shoulder, mid_shoulder_available, mid_shoulder_conf = compute_midpoint_with_conf(
        l_shoulder_lm, r_shoulder_lm, l_shoulder_vis, r_shoulder_vis
    )
    mid_shoulder_reliable = bool(
        mid_shoulder_available and mid_shoulder_conf >= SHOULDER_CONF_THR
    )

    mid_hip, mid_hip_available, mid_hip_conf = compute_midpoint_with_conf(
        l_hip_lm, r_hip_lm, l_hip_vis, r_hip_vis
    )
    mid_hip_reliable = bool(
        mid_hip_available and mid_hip_conf >= HIP_CONF_THR
    )
    trunk_support_reliable = bool(mid_shoulder_reliable and mid_hip_reliable)

    head_center, head_center_source, head_center_conf, head_center_z_reliable = compute_head_center(
        _as_landmark(nose), _as_landmark(l_eye), _as_landmark(r_eye),
        _as_landmark(l_ear), _as_landmark(r_ear),
        nose_vis=nose_vis,
        l_eye_vis=l_eye_vis, r_eye_vis=r_eye_vis,
        l_ear_vis=l_ear_vis, r_ear_vis=r_ear_vis,
    )

    upper_z_reliable = bool(head_center_z_reliable and mid_shoulder_reliable)

    features = extract_features_as_dict(
        nose, l_eye, r_eye, l_ear, r_ear,
        l_shoulder, r_shoulder,
        l_hip=l_hip, r_hip=r_hip,
        nose_vis=nose_vis,
        l_eye_vis=l_eye_vis, r_eye_vis=r_eye_vis,
        l_ear_vis=l_ear_vis, r_ear_vis=r_ear_vis,
        l_shoulder_vis=l_shoulder_vis, r_shoulder_vis=r_shoulder_vis,
        l_hip_vis=l_hip_vis, r_hip_vis=r_hip_vis,
    )

    control = {
        "head_center_source":      head_center_source,
        "head_center_conf":        float(head_center_conf),
        "head_center_z_reliable":  bool(head_center_z_reliable),
        "mid_shoulder_conf":       float(mid_shoulder_conf),
        "mid_shoulder_reliable":   bool(mid_shoulder_reliable),
        "upper_z_reliable":        bool(upper_z_reliable),
        "trunk_support_reliable":  bool(trunk_support_reliable),
    }

    return features, control


print(f"Utility functions loaded — {len(FEATURE_NAMES)} features.")
print(f"Features: {FEATURE_NAMES}")

# **3. ML Model Training**

In this section we train and compare different classification models
based on the postural features extracted from the dataset.

The tested models are:
- **Random Forest**
- **MLP (Multi-Layer Perceptron)**
- **SVM (Support Vector Machine)**
- **KNN (K-Nearest Neighbors)**
- **Gradient Boosting**

For each model we evaluate: accuracy, per-class F1-score, confusion matrix.
The best model will be saved along with the scaler and label encoder
to be used in the real-time inference phase with MediaPipe.

## Import libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

## Loading the feature dataset

In [ ]:
df = pd.read_csv("posture_features.csv")
print("Dataset shape:", df.shape)
print(df.head())

## Selection of Input Features

In [ ]:
PRIMARY_FEATURES = [
    "head_tilt",
    "shoulder_slope",
    "head_lateral_ratio",
    "head_y_ratio",
    "head_neck_vertical_angle",
    "head_shoulder_alignment",
    "upper_body_inclination",
    "head_trunk_diff",
]

SUPPORT_FEATURES = [
    "trunk_forward_backward_angle",
    "shoulder_z_asym",
    "hip_shoulder_lateral",
    "ear_height_asym",
]

ALL_FEATURES = PRIMARY_FEATURES + SUPPORT_FEATURES

FEATURE_SETS = {
    "primary_only":        PRIMARY_FEATURES,
    "primary_plus_support": ALL_FEATURES,
}

required_columns = ["subject", "upperbody_label"] + ALL_FEATURES
missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns in CSV: {missing_required}")

print("Primary features:", len(PRIMARY_FEATURES))
print(PRIMARY_FEATURES)
print("\nSupport features:", len(SUPPORT_FEATURES))
print(SUPPORT_FEATURES)
print("\nAll features:", len(ALL_FEATURES))

tfba = df["trunk_forward_backward_angle"].dropna()
print(f"\ntrunk_forward_backward_angle — range effettivo: [{tfba.min():.1f}°, {tfba.max():.1f}°]")
print(f"mediana: {tfba.median():.2f}°  (atteso ~0° per postura neutra)")
print(f"NaN: {df['trunk_forward_backward_angle'].isna().sum()} / {len(df)}")

for feat in ["shoulder_z_asym", "hip_shoulder_lateral", "ear_height_asym"]:
    s = df[feat].dropna()
    print(f"{feat}: mean={s.mean():.3f}, std={s.std():.3f}, NaN={df[feat].isna().sum()}")

## Dataset Cleaning and Target Definition

In [ ]:
# Keep only rows with valid ID/label
df_clean = df.dropna(subset=["subject", "upperbody_label"]).copy()

# For primary features, require completeness
df_clean = df_clean.dropna(subset=PRIMARY_FEATURES).copy()

print(
    f"Rows before cleaning: {len(df)} | "
    f"Rows after cleaning: {len(df_clean)} | "
    f"Removed: {len(df) - len(df_clean)}"
)

if df_clean.empty:
    raise ValueError("The cleaned dataset is empty.")

# Target and groups
y = df_clean["upperbody_label"].values
groups = df_clean["subject"].values

print("\nClass distribution:")
print(df_clean["upperbody_label"].value_counts())

print("\nSubjects:")
print(sorted(df_clean["subject"].unique()))

## Label Encoding

In [ ]:
label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)

print("Classes:", list(label_encoder.classes_))
print("Class distribution:", dict(zip(label_encoder.classes_, np.bincount(y_enc))))

## Evaluation Function

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te, label_encoder):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    macro_f1 = f1_score(y_te, y_pred, average="macro")

    print("\n" + ("=" * 70))
    print(f"{name}")
    print("=" * 70)
    print(f"Accuracy : {acc:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}")
    print()
    print(classification_report(y_te, y_pred, target_names=label_encoder.classes_))

    cm = confusion_matrix(y_te, y_pred)

    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
    )
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    return {
        "model": model,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "y_pred": y_pred,
    }

## Split by subject

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_clean, y_enc, groups=groups))

train_subjects = set(groups[train_idx])
test_subjects = set(groups[test_idx])

assert len(train_subjects & test_subjects) == 0, "Leak detected: shared subjects found."

print("Subjects in train:", sorted(train_subjects))
print("Subjects in test:", sorted(test_subjects))
print(f"Train samples: {len(train_idx)} | Test samples: {len(test_idx)}")

print("\nTest label distribution:")
print(pd.Series(label_encoder.inverse_transform(y_enc[test_idx])).value_counts())

## Results containers

In [ ]:
# Results containers
results = []
trained_models = []

print("\nML setup ready.")

## Definition of the models

In [ ]:
def build_models():
    models = {
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            ))
        ]),

        "Gradient Boosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", GradientBoostingClassifier(
                n_estimators=200,
                learning_rate=0.1,
                max_depth=4,
                random_state=42
            ))
        ]),

        "SVM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                C=10,
                gamma="scale",
                probability=True,
                random_state=42
            ))
        ]),

        "KNN": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(
                n_neighbors=7,
                weights="distance",
                metric="euclidean",
                n_jobs=-1
            ))
        ]),

        "MLP": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(256, 128, 64),
                activation="relu",
                max_iter=500,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=42
            ))
        ]),
    }
    return models

## Training block 

In [ ]:
for feature_set_name, feature_cols in FEATURE_SETS.items():
    print("\n" + "#" * 80)
    print(f"FEATURE SET: {feature_set_name}")
    print("#" * 80)

    X = df_clean[feature_cols].copy().values
    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y_enc[train_idx]
    y_test = y_enc[test_idx]

    print(f"Using {len(feature_cols)} features")
    print(feature_cols)

    models = build_models()

    for model_name, model in models.items():
        full_name = f"{model_name} [{feature_set_name}]"

        result = evaluate(
            name=full_name,
            model=model,
            X_tr=X_train,
            y_tr=y_train,
            X_te=X_test,
            y_te=y_test,
            label_encoder=label_encoder
        )

        results.append({
            "feature_set": feature_set_name,
            "model_name": model_name,
            "full_name": full_name,
            "n_features": len(feature_cols),
            "accuracy": result["accuracy"],
            "macro_f1": result["macro_f1"],
        })

        trained_models.append({
            "full_name": full_name,
            "model": result["model"],
            "feature_cols": feature_cols,
        })

## Final ranking

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by=["macro_f1", "accuracy"],
    ascending=False
).reset_index(drop=True)

print("FINAL RANKING:")
print(results_df[["full_name", "n_features", "accuracy", "macro_f1"]])

## Best model selection

In [ ]:
results_support = results_df[
    results_df["feature_set"] == "primary_plus_support"
].copy().sort_values(by=["macro_f1", "accuracy"], ascending=False).reset_index(drop=True)

if results_support.empty:
    raise ValueError("Nessun modello primary_plus_support trovato.")

best_row        = results_support.iloc[0]
best_model_name = best_row["full_name"]

best_model_entry  = next(m for m in trained_models if m["full_name"] == best_model_name)
best_model        = best_model_entry["model"]
best_feature_cols = best_model_entry["feature_cols"]

print("[DEPLOY MODE: PRIMARY + SUPPORT]")
print(f"Best model:    {best_model_name}")
print(f"Best accuracy: {best_row['accuracy']:.4f}")
print(f"Best macro F1: {best_row['macro_f1']:.4f}")
print(f"Features ({len(best_feature_cols)}): {best_feature_cols}")

print("\nTop 3 support models:")
print(results_support.head(3)[["full_name", "accuracy", "macro_f1"]])

with open("best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)
with open("best_feature_columns.pkl", "wb") as f:
    pickle.dump(best_feature_cols, f)

print("\nSaved: best_model.pkl | label_encoder.pkl | best_feature_columns.pkl")

# 4. Final Model Selection and Real-Time Inference with MediaPipe

In this part of the notebook we complete two fundamental steps of the pipeline:

1. **Final model comparison on group split**
2. **Loading the best model for real-time webcam inference**

The goal is to use the same approach from training in real time:
- landmark extraction with **MediaPipe Pose**
- postural feature construction
- normalization with the saved scaler
- posture prediction using the best model
- result visualization on the frame

## Import libraries for webcam and inference

In [ ]:
import os
import cv2
import json
import pickle
import collections
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import time
from datetime import datetime
from IPython.display import display

## Loading the saved model

In [ ]:
required_files = ["best_model.pkl", "label_encoder.pkl", "best_feature_columns.pkl"]
missing_files  = [f for f in required_files if not os.path.exists(f)]
if missing_files:
    raise FileNotFoundError(f"Missing files: {missing_files}")
 
with open("best_model.pkl", "rb") as f:
    best_model = pickle.load(f)
with open("label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)
with open("best_feature_columns.pkl", "rb") as f:
    BEST_FEATURE_COLUMNS = pickle.load(f)
 
print("Model:    ", type(best_model).__name__)
print("Classes:  ", list(label_encoder.classes_))
print("Features: ", BEST_FEATURE_COLUMNS)

## Inizialization of MediaPipe Pose

In [ ]:
mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
 
if "pose" in dir() and pose is not None:
    try:
        pose.close()
    except Exception:
        pass
 
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)
print("MediaPipe Pose inizialized.")

## Iphone connection

In [ ]:
CAMERA_INDEX = 1

cap = cv2.VideoCapture(CAMERA_INDEX)
# prova una risoluzione più leggera
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

actual_w = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
actual_h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

print(f"Acquired resolution: {actual_w}x{actual_h}")

## Main pipeline parameters

In [ ]:
CONF_THRESHOLD     = 0.5
SMOOTH_WINDOW      = 10
CALIBRATION_FRAMES = 30
POSTURE_FRAMES     = 15
REQUIRED_IDX       = [0, 11, 12]   

z_buffer = collections.deque(maxlen=SMOOTH_WINDOW)

FEATURE_NAMES = [
    "head_tilt",
    "shoulder_slope",
    "head_lateral_ratio",
    "head_y_ratio",
    "head_neck_vertical_angle",
    "head_shoulder_alignment",
    "upper_body_inclination",
    "head_trunk_diff",
    "trunk_forward_backward_angle",
    "shoulder_z_asym",
    "hip_shoulder_lateral",
    "ear_height_asym",
]
PRIMARY_FEATURES = FEATURE_NAMES[:8]
SUPPORT_FEATURES = FEATURE_NAMES[8:]

# Label colors and descriptions
LABEL_COLORS = {
    "TUP": (0, 220, 0),
    "TLL": (255, 100, 0),
    "TLR": (255, 100, 0),
    "TLF": (0, 180, 255),
    "TLB": (180, 0, 255),
}
LABEL_DESC = {
    "TUP": "Correct posture",
    "TLL": "Left lateral tilt",
    "TLR": "Right lateral tilt",
    "TLF": "Forward head / slumping",
    "TLB": "Backward posture",
}

# Feature alignment check
if set(FEATURE_NAMES) != set(BEST_FEATURE_COLUMNS):
    print("   FEATURE_NAMES and BEST_FEATURE_COLUMNS do not match!")
    print("   Missing:", set(BEST_FEATURE_COLUMNS) - set(FEATURE_NAMES))
    print("   Extra:  ", set(FEATURE_NAMES) - set(BEST_FEATURE_COLUMNS))
else:
    print(f"✓  Features aligned — {len(FEATURE_NAMES)} features.")

## Preprocessing and feature extraction functions

In [ ]:
def capture_frames_batch(n_frames):
    frames = []
    for _ in range(n_frames):
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
        time.sleep(0.12)
    return frames
 
 
def landmarks_to_array(results):
    lm = results.pose_landmarks.landmark
    return np.array([[l.x, l.y, l.z] for l in lm], dtype=float)
 
 
def smooth_z(lm_array):
    z_buffer.append(lm_array[:, 2].copy())
    smoothed = lm_array.copy()
    smoothed[:, 2] = np.mean(z_buffer, axis=0)
    return smoothed
 
 
def check_visibility(results):
    lm = results.pose_landmarks.landmark
    shoulders_ok = (lm[11].visibility >= CONF_THRESHOLD and
                    lm[12].visibility >= CONF_THRESHOLD)
    head_ok = ((lm[2].visibility >= CONF_THRESHOLD and
                lm[5].visibility >= CONF_THRESHOLD) or
               (lm[7].visibility >= CONF_THRESHOLD and
                lm[8].visibility >= CONF_THRESHOLD) or
               lm[0].visibility >= CONF_THRESHOLD)
    return shoulders_ok and head_ok
 
 
def extract_features_from_array(lm_array, results):
    lm = results.pose_landmarks.landmark
 
    features_dict, control_dict = extract_features_with_control(
        nose=lm_array[0],
        l_eye=lm_array[2],  r_eye=lm_array[5],
        l_ear=lm_array[7],  r_ear=lm_array[8],
        l_shoulder=lm_array[11], r_shoulder=lm_array[12],
        l_hip=lm_array[23],      r_hip=lm_array[24],
        nose_vis=lm[0].visibility,
        l_eye_vis=lm[2].visibility,  r_eye_vis=lm[5].visibility,
        l_ear_vis=lm[7].visibility,  r_ear_vis=lm[8].visibility,
        l_shoulder_vis=lm[11].visibility, r_shoulder_vis=lm[12].visibility,
        l_hip_vis=lm[23].visibility,      r_hip_vis=lm[24].visibility,
    )
 
    # Flag extra per il debug overlay
    control_dict["shoulders_reliable"] = (lm[11].visibility >= CONF_THRESHOLD and
                                           lm[12].visibility >= CONF_THRESHOLD)
    control_dict["head_reliable"] = (
        (lm[2].visibility >= CONF_THRESHOLD and lm[5].visibility >= CONF_THRESHOLD) or
        (lm[7].visibility >= CONF_THRESHOLD and lm[8].visibility >= CONF_THRESHOLD) or
        lm[0].visibility >= CONF_THRESHOLD
    )
    control_dict["hips_reliable"] = (lm[23].visibility >= CONF_THRESHOLD and
                                      lm[24].visibility >= CONF_THRESHOLD)
    control_dict["trunk_support_reliable"] = control_dict["hips_reliable"]
 
    return features_dict, control_dict
 
 
def feature_dict_to_model_vector(feature_dict, feature_columns):
    return np.array([feature_dict.get(f, np.nan) for f in feature_columns], dtype=float)

## Prediction and classification functions

### Constants and baseline helpers

_safe_std: during calibration the user might stay almost perfectly still, producing a very low or zero std. This function ensures that each feature always has a different "minimum sensitivity" by type: for example head_lateral_ratio is already very sensitive (floor 0.08), while upper_body_inclination requires larger deviations (floor 4.0) before triggering the alarm.

_clamped_delta: answers the question: "how much has this feature shifted compared to the calibrated neutral posture?" — in standard deviation units. The clamp at ±4 prevents an extreme value (e.g. head completely turned) from producing a huge score and destabilizing the classification.

get_baseline_value: reads the reference value (median by default) from the calibrated baseline for a feature. Used every time we need to compare the current posture with the user's neutral one.

get_baseline_std: reads the user's natural variability for a feature from the baseline. Used to determine how much a feature needs to shift to be considered "anomalous" compared to how that person normally moves.

In [ ]:
EPS_STD = 1e-3

# Minimum std floor per feature to avoid zero/near-zero sensitivity
STD_FLOOR = {
    "head_tilt":                    3.5,
    "shoulder_slope":               3.5,
    "head_lateral_ratio":           0.06,
    "head_y_ratio":                 0.06,
    "head_neck_vertical_angle":     1.5,
    "head_shoulder_alignment":      5.0,
    "upper_body_inclination":       3.0,
    "head_trunk_diff":              5.0,
    "trunk_forward_backward_angle": 4.0,
    "shoulder_z_asym":              0.25,
    "hip_shoulder_lateral":         0.15,
    "ear_height_asym":              0.04,
}
DELTA_CLAMP = 4.0
HYSTERESIS = 0.25
TUP_FLOOR = 1.4


def _safe_std(std, feature_name):
    """Return std with a per-feature floor to prevent near-zero sensitivity."""
    floor = STD_FLOOR.get(feature_name, 0.001)
    return max(std, floor) if (np.isfinite(std) and std > 0) else floor


def _clamped_delta(val, baseline, std):
    """Compute deviation from baseline in std units, clamped to ±DELTA_CLAMP."""
    if not (np.isfinite(val) and np.isfinite(baseline) and
            np.isfinite(std) and std > EPS_STD):
        return np.nan
    return float(np.clip((val - baseline) / std, -DELTA_CLAMP, DELTA_CLAMP))


def get_baseline_value(baseline_stats, name, field="median"):
    """Retrieve the calibrated reference value (default: median) for a feature."""
    if baseline_stats is None or name not in baseline_stats:
        return np.nan
    return baseline_stats[name].get(field, np.nan)


def get_baseline_std(baseline_stats, name):
    """Retrieve the calibrated natural variability (std) for a feature."""
    if baseline_stats is None or name not in baseline_stats:
        return np.nan
    return baseline_stats[name].get("std", np.nan)

### Predict and score helpers

predict: passes the feature vector to the trained ML model and returns the predicted class (e.g. "TLF"), its probability, and all per-class probabilities. This is the point where the model "votes" on the current posture.

_compute_lateral_score: measures how much the user is tilting laterally compared to their calibrated neutral posture, combining two signals: the horizontal head displacement (head_lateral_ratio, higher weight) and the torso inclination (upper_body_inclination, lower weight). A positive value indicates left tilt (TLL), negative indicates right tilt (TLR).

_compute_sagittal_score: measures how much the user is leaning forward or backward compared to their calibrated neutral posture, combining three signals: the trunk depth inclination (trunk_forward_backward_angle, main weight), the relative head height with respect to the shoulders (head_y_ratio), and the head-neck angle (head_neck_vertical_angle, used only as reinforcement if it agrees with the other two). A negative value indicates forward posture (TLF), positive indicates backward posture (TLB).

In [ ]:
# ── Lateral sign ──────────────────────────────────────────────────────────
# With the phone at 30/45° angle, shoulder_z_asym is the main signal.
# The sign depends on which side the phone is on:
#   phone on your LEFT  →  LAT_SIGN = +1
#   phone on your RIGHT →  LAT_SIGN = -1
LAT_SIGN = +1


def predict(features_vector, model, label_encoder):
    """Run the trained model on a feature vector and return predicted class, confidence, and all probabilities."""
    fv    = np.asarray(features_vector, dtype=float).reshape(1, -1)
    probs = model.predict_proba(fv)[0]
    idx   = int(np.argmax(probs))
    return label_encoder.inverse_transform([idx])[0], float(probs[idx]), probs


def _compute_lateral_score(feat, baseline_stats):
    """
    Lateral score optimized for oblique framing (phone at 30/45°).

    With a side-angled camera, head_lateral_ratio measures displacement
    on the image plane (not the body's left/right axis) and is unreliable.
    shoulder_z_asym instead measures shoulder depth asymmetry: when tilting
    laterally one shoulder moves closer and the other farther from the
    camera — a robust signal regardless of viewing angle.

    Weights:
      shoulder_z_asym     3.0  — primary signal (shoulder depth)
      ear_height_asym     1.5  — relative ear height (reliable)
      head_lateral_ratio  0.5  — reduced weight, noisy at oblique angles
    """
    pairs = [
        ("shoulder_z_asym",    3.0),
        ("ear_height_asym",    1.5),
        ("head_lateral_ratio", 0.5),
    ]
    components = []
    for name, w in pairs:
        val = feat.get(name, np.nan)
        if not np.isfinite(val):
            continue
        b   = get_baseline_value(baseline_stats, name)
        std = _safe_std(get_baseline_std(baseline_stats, name), name)
        d   = _clamped_delta(val, b, std)
        if np.isfinite(d):
            components.append((d, w))
    if not components:
        return np.nan
    total_w = sum(w for _, w in components)
    raw = sum(v * w for v, w in components) / total_w
    return float(raw * LAT_SIGN)


def _compute_sagittal_score(feat, baseline_stats):
    """
    Sagittal score.
    Convention: negative = TLF (forward), positive = TLB (backward).

    Signals:
      trunk_forward_backward_angle (weight 3.0) — primary
        Forward lean increases trunk_fba relative to baseline.
        Negated so that forward lean yields a negative score (TLF).

      head_y_ratio (weight 1.5) — secondary
        In image coordinates y grows downward, so head_y_ratio is positive
        when the head is above the shoulders (upright). Forward lean drops
        the head, decreasing head_y_ratio → negative delta → negative
        contribution, consistent with TLF convention.

      head_neck_vertical_angle (weight 1.0) — reinforcement, only if
        it agrees with head_y_ratio (concordance check below).
    """
    components = []

    # trunk_forward_backward_angle — primary signal
    # Delta is negated: forward lean = negative score (TLF)
    tfb = feat.get("trunk_forward_backward_angle", np.nan)
    if np.isfinite(tfb):
        b   = get_baseline_value(baseline_stats, "trunk_forward_backward_angle")
        std = _safe_std(get_baseline_std(baseline_stats, "trunk_forward_backward_angle"),
                        "trunk_forward_backward_angle")
        d   = _clamped_delta(tfb, b, std)
        if np.isfinite(d):
            components.append((d, 3.0))

    # head_y_ratio — secondary signal
    # Head drops on forward lean → delta < 0 → negative contribution (TLF)
    hyr = feat.get("head_y_ratio", np.nan)
    if np.isfinite(hyr):
        b   = get_baseline_value(baseline_stats, "head_y_ratio")
        std = _safe_std(get_baseline_std(baseline_stats, "head_y_ratio"), "head_y_ratio")
        d   = _clamped_delta(hyr, b, std)
        if np.isfinite(d):
            components.append((d, 1.5))

    # head_neck_vertical_angle — reinforcement only if it agrees with hyr
    # hnv rises when the head drops forward → d > 0
    # hyr drops on forward lean → hyr_d < 0
    hnv = feat.get("head_neck_vertical_angle", np.nan)
    if np.isfinite(hnv) and hnv > 3.0 and np.isfinite(hyr):
        b   = get_baseline_value(baseline_stats, "head_neck_vertical_angle")
        std = _safe_std(get_baseline_std(baseline_stats, "head_neck_vertical_angle"),
                        "head_neck_vertical_angle")
        d   = _clamped_delta(hnv, b, std)
        if np.isfinite(d):
            hyr_d = _clamped_delta(
                hyr,
                get_baseline_value(baseline_stats, "head_y_ratio"),
                _safe_std(get_baseline_std(baseline_stats, "head_y_ratio"), "head_y_ratio")
            )
            if np.isfinite(hyr_d) and d * hyr_d < 0:
                components.append((-d, 1.0))

    if not components:
        return np.nan
    total_w = sum(w for _, w in components)
    return sum(v * w for v, w in components) / total_w

### Classify posture

classify_posture: the core of the classification system. It takes the ML model prediction and "corrects" or "confirms" it using geometric rules based on the calibrated baseline. The idea is that the ML alone can fail in edge cases, so geometric rules act as a safeguard.

1. Extreme lateral guard (lat_abs > 3.5)
If the head is displaced laterally by a very large amount, the geometric rule overrides the ML without even checking its confidence. The case is so obvious that trusting the model is unnecessary.
2. High-confidence ML (confidence >= 0.85)
If the model is very confident in its prediction, no intervention is applied. Geometric rules only step in when the ML is uncertain.
3. Geometric TLR (head_tilt < -30° and head_shoulder_alignment < -15°)
Detects the specific case where the user is filmed from the side and the head appears rotated/tilted in an extreme way — a situation where the ML tends to get confused but the geometry is clear.
4. Geometric TLF (hnv > baseline + 8° and hyr < baseline - 0.08)
Detects forward head drop: the neck-vertical angle has increased relative to the baseline and the head has dropped relative to the shoulders. This block does not trigger if there is also a strong lateral tilt (to avoid false TLF when the user is simply leaning sideways).
5. No signal (lat_sig = False and sag_sig = False)
If neither the lateral nor the sagittal signal exceeds the threshold, the posture is considered correct (TUP) regardless of what the ML says.
6. Strong lateral
If the lateral signal clearly exceeds the threshold, TLL or TLR is returned based on the sign of the score.
7. Lateral dominates sagittal
If both signals are present but the lateral one is at least 1.1x stronger than the sagittal, lateral wins. This prevents a small sagittal component from changing the classification when lateral tilt is the main issue.
8. Sagittal only
If there is only a sagittal signal with no relevant lateral component, TLF (forward) or TLB (backward) is returned based on the sign.
9. Ambiguous case
If no rule produced a clear answer, the ML prediction is used as a fallback.

In [ ]:
def classify_posture(label_ml, confidence, feature_dict, baseline_stats,
                     current_label="TUP"):
    """
    Hybrid classifier: ML prediction + geometric rules.
    """
    ML_TRUST_THR = 0.98
    SAG_ABS_THR_ENTER = 2.2
    LAT_ABS_THR_ENTER_LEFT = 2.0
    LAT_ABS_THR_ENTER_RIGHT = 1.6
    DOMINANCE_RATIO = 1.15
    SAGITTAL_HOLD_RATIO = 1.35
    LATERAL_HOLD_RATIO = 1.45
    TLB_LAT_RATIO = 1.6

    trunk_fba = feature_dict.get("trunk_forward_backward_angle")
    trunk_fba_available = trunk_fba is not None and np.isfinite(trunk_fba)
    if not trunk_fba_available:
        SAG_ABS_THR_ENTER = max(SAG_ABS_THR_ENTER, 2.4)

    lat_score = _compute_lateral_score(feature_dict, baseline_stats)
    sag_score = _compute_sagittal_score(feature_dict, baseline_stats)
    lat_abs = abs(lat_score) if np.isfinite(lat_score) else 0.0
    sag_abs = abs(sag_score) if np.isfinite(sag_score) else 0.0

    # Select lateral entry threshold based on current label and score direction
    if current_label == "TLR":
        lat_enter_base = LAT_ABS_THR_ENTER_RIGHT
    elif current_label == "TLL":
        lat_enter_base = LAT_ABS_THR_ENTER_LEFT
    elif np.isfinite(lat_score) and lat_score < 0:
        lat_enter_base = LAT_ABS_THR_ENTER_RIGHT
    else:
        lat_enter_base = LAT_ABS_THR_ENTER_LEFT

    LAT_ABS_THR_ENTER = lat_enter_base
    LAT_ABS_THR_EXIT = LAT_ABS_THR_ENTER * (1.0 - HYSTERESIS)
    SAG_ABS_THR_EXIT = SAG_ABS_THR_ENTER * (1.0 - HYSTERESIS)

    # Raise sagittal threshold when a moderate lateral signal is present
    if np.isfinite(lat_score) and lat_abs > 1.5:
        SAG_ABS_THR_ENTER = max(SAG_ABS_THR_ENTER, 2.5)
        SAG_ABS_THR_EXIT = SAG_ABS_THR_ENTER * (1.0 - HYSTERESIS)

    # Dominance ratios: higher when trying to exit the current state (hysteresis)
    lat_over_sag_ratio = (
        SAGITTAL_HOLD_RATIO if current_label in ("TLF", "TLB")
        else DOMINANCE_RATIO
    )

    sag_over_lat_ratio = (
        LATERAL_HOLD_RATIO if current_label in ("TLL", "TLR")
        else DOMINANCE_RATIO
    )
    
    if current_label in ("TLL", "TLR"):
        eff_lat_thr = LAT_ABS_THR_EXIT
        eff_sag_thr = SAG_ABS_THR_ENTER
    elif current_label in ("TLF", "TLB"):
        eff_lat_thr = LAT_ABS_THR_ENTER + 0.25
        eff_sag_thr = SAG_ABS_THR_EXIT
    else:
        eff_lat_thr = LAT_ABS_THR_ENTER
        eff_sag_thr = SAG_ABS_THR_ENTER

    lat_sig = lat_abs > eff_lat_thr
    sag_sig = sag_abs > eff_sag_thr
    lat_weak = lat_abs < LAT_ABS_THR_EXIT
    sag_weak = sag_abs < SAG_ABS_THR_EXIT

    info = {
        "lateral_score": round(float(lat_score), 4) if np.isfinite(lat_score) else None,
        "sagittal_score": round(float(sag_score), 4) if np.isfinite(sag_score) else None,
        "eff_lat_thr": round(eff_lat_thr, 3),
        "eff_sag_thr": round(eff_sag_thr, 3),
        "rule_triggered": False,
        "confidence_tier": "high",
        "reason": [],
    }

    # ── TUP floor — both below minimum threshold → confident TUP ─
    if lat_abs < TUP_FLOOR and sag_abs < TUP_FLOOR:
        info["reason"].append(
            f"TUP floor (lat={lat_abs:.2f} sag={sag_abs:.2f} < {TUP_FLOOR})"
        )
        return "TUP", info

    # ── Extreme lateral guard ─────────────────────────────────────
    if np.isfinite(lat_score) and lat_abs > 3.5:
        info["rule_triggered"] = True
        info["reason"].append(f"extreme lateral ({lat_abs:.2f}) — ML overridden")
        return ("TLL" if lat_score > 0 else "TLR"), info

    # ── TLB protection ────────────────────────────────────────────
    if current_label == "TLB" and sag_sig:
        if lat_sig and lat_abs > sag_abs * TLB_LAT_RATIO:
            info["rule_triggered"] = True
            direction = "TLL" if lat_score > 0 else "TLR"
            info["reason"].append(
                f"TLB→lateral dominates clearly ({lat_abs:.2f} > {sag_abs:.2f}×{TLB_LAT_RATIO})"
            )
            return direction, info
        else:
            info["rule_triggered"] = True
            info["reason"].append(f"TLB holds (sag={sag_abs:.2f} lat={lat_abs:.2f})")
            return "TLB", info

    # ── High-confidence ML ────────────────────────────────────────
    if confidence >= ML_TRUST_THR:
        info["reason"].append(f"high ML confidence ({confidence:.1%})")
        return label_ml, info

    tfb = feature_dict.get("trunk_forward_backward_angle", np.nan)
    hyr = feature_dict.get("head_y_ratio", np.nan)
    hnv = feature_dict.get("head_neck_vertical_angle", np.nan)

    b_tfb = get_baseline_value(baseline_stats, "trunk_forward_backward_angle")
    b_hyr = get_baseline_value(baseline_stats, "head_y_ratio")
    b_hnv = get_baseline_value(baseline_stats, "head_neck_vertical_angle")

    tfb_delta = tfb - b_tfb if np.isfinite(tfb) and np.isfinite(b_tfb) else np.nan
    hyr_delta = hyr - b_hyr if np.isfinite(hyr) and np.isfinite(b_hyr) else np.nan
    hnv_delta = hnv - b_hnv if np.isfinite(hnv) and np.isfinite(b_hnv) else np.nan

    # ── Geometric TLB ─────────────────────────────────────────────
    if (
        np.isfinite(tfb_delta) and tfb_delta > 8.0 and
        sag_abs > 1.6 and
        lat_abs < LAT_ABS_THR_ENTER_LEFT and
        (
            (np.isfinite(hyr_delta) and hyr_delta > 0.05) or
            (np.isfinite(hnv_delta) and hnv_delta < -3.0)
        )
    ):
        info["rule_triggered"] = True
        info["reason"].append(
            f"geometric TLB (tfb_δ={tfb_delta:.1f}°, hyr_δ={hyr_delta:.3f}, hnv_δ={hnv_delta:.1f}°)"
        )
        return "TLB", info

    # ── Geometric TLF ─────────────────────────────────────────────
    if (
        np.isfinite(feature_dict.get("head_neck_vertical_angle", np.nan)) and
        np.isfinite(feature_dict.get("head_y_ratio", np.nan))
    ):
        hnv = feature_dict["head_neck_vertical_angle"]
        hyr = feature_dict["head_y_ratio"]
        b_hnv = get_baseline_value(baseline_stats, "head_neck_vertical_angle")
        b_hyr = get_baseline_value(baseline_stats, "head_y_ratio")
        hnv_delta = hnv - b_hnv if np.isfinite(b_hnv) else np.nan
        hyr_delta = hyr - b_hyr if np.isfinite(b_hyr) else np.nan

        if (
            np.isfinite(hnv_delta) and hnv_delta > 10.0 and
            np.isfinite(hyr_delta) and hyr_delta < -0.10 and
            lat_abs < 2.2 and
            not (np.isfinite(tfb_delta) and tfb_delta > 2.0)
        ):
            info["rule_triggered"] = True
            info["reason"].append(
                f"geometric TLF (hnv_δ={hnv_delta:.1f}°, hyr_δ={hyr_delta:.3f})"
            )
            return "TLF", info

    # ── Both signals weak → TUP ───────────────────────────────────
    if lat_weak and sag_weak:
        info["reason"].append("weak geometric signals → TUP")
        return "TUP", info

    # ── Strong lateral ────────────────────────────────────────────
    if lat_sig and (not sag_sig or lat_abs >= sag_abs * lat_over_sag_ratio):
        info["rule_triggered"] = True
        direction = "TLL" if lat_score > 0 else "TLR"
        info["reason"].append(f"lateral ({lat_abs:.2f} > thr={eff_lat_thr:.2f})")
        return direction, info

    # ── Sagittal only ─────────────────────────────────────────────
    if sag_sig and not lat_sig:
        info["rule_triggered"] = True
        direction = "TLB" if sag_score > 0 else "TLF"
        info["reason"].append(f"sagittal ({sag_abs:.2f} > thr={eff_sag_thr:.2f})")
        return direction, info

    # ── Both signals present ──────────────────────────────────────
    if lat_sig and sag_sig:
        if lat_abs >= sag_abs * lat_over_sag_ratio:
            info["rule_triggered"] = True
            direction = "TLL" if lat_score > 0 else "TLR"
            info["reason"].append(f"lateral dominates ({lat_abs:.2f} > {sag_abs:.2f})")
            return direction, info
        elif sag_abs > lat_abs * sag_over_lat_ratio:
            info["rule_triggered"] = True
            direction = "TLB" if sag_score > 0 else "TLF"
            info["reason"].append(f"sagittal dominates ({sag_abs:.2f} > {lat_abs:.2f})")
            return direction, info
        else:
            info["rule_triggered"] = True

            # Ambiguous: hold current label if already in a non-TUP state
            if current_label in ("TLF", "TLB"):
                info["reason"].append(
                    f"ambiguous → holding {current_label} (sag={sag_abs:.2f} lat={lat_abs:.2f})"
                )
                return current_label, info

            if current_label in ("TLL", "TLR"):
                info["reason"].append(
                    f"ambiguous → holding {current_label} (sag={sag_abs:.2f} lat={lat_abs:.2f})"
                )
                return current_label, info

            # No prior state: pick the stronger axis
            if sag_abs >= lat_abs:
                direction = "TLB" if sag_score > 0 else "TLF"
                info["reason"].append(f"ambiguous → sagittal ({sag_abs:.2f} >= {lat_abs:.2f})")
            else:
                direction = "TLL" if lat_score > 0 else "TLR"
                info["reason"].append(f"ambiguous → lateral ({lat_abs:.2f} > {sag_abs:.2f})")

            return direction, info

    # ── No geometric signal → TUP ─────────────────────────────────
    info["reason"].append("no signal → TUP")
    return "TUP", info

## Drawing functions for the debug overlay

In [ ]:
GREEN  = (0, 220, 0)
YELLOW = (0, 220, 255)
ORANGE = (0, 140, 255)
RED    = (0, 0, 255)
WHITE  = (255, 255, 255)
BLACK  = (0, 0, 0)
CYAN   = (255, 220, 0)
GRAY   = (180, 180, 180)
 
 
def draw_landmark_point(frame, lm, idx, color, radius=6):
    h, w = frame.shape[:2]
    x = int(lm[idx].x * w)
    y = int(lm[idx].y * h)
    cv2.circle(frame, (x, y), radius, color, -1)
    cv2.circle(frame, (x, y), radius + 2, WHITE, 1)
    return (x, y)
 
 
def draw_line_between(frame, p1, p2, color, thickness=2):
    cv2.line(frame, p1, p2, color, thickness)
 
 
def put_text_pro(frame, text, pos, scale=0.7, color=WHITE, thickness=1):
    cv2.putText(frame, text, pos, cv2.FONT_HERSHEY_DUPLEX,
                scale, BLACK, thickness + 2, cv2.LINE_AA)
    cv2.putText(frame, text, pos, cv2.FONT_HERSHEY_DUPLEX,
                scale, color, thickness, cv2.LINE_AA)
 
 
def draw_debug_overlay(frame, lm, feature_dict, label, confidence,
                        confidence_tier="high", rule_info=None):
    h, w   = frame.shape[:2]
    color  = LABEL_COLORS.get(label, WHITE)
    desc   = LABEL_DESC.get(label, label)
 
    # Barra superiore
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 80), (10, 10, 10), -1)
    cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)
 
    put_text_pro(frame, f"{label}  {desc}", (14, 30),
                 scale=0.85, color=color, thickness=2)
    put_text_pro(frame, f"Conf: {confidence:.1%}", (14, 56),
                 scale=0.55, color=GRAY, thickness=1)
 
    if rule_info and rule_info.get("reason"):
        reason_str = " | ".join(rule_info["reason"])[:60]
        put_text_pro(frame, reason_str, (14, 74),
                     scale=0.38, color=YELLOW, thickness=1)
 
    # Barra laterale verde/rossa
    bar_color = GREEN if label == "TUP" else RED
    cv2.rectangle(frame, (w - 10, 0), (w, h), bar_color, -1)
 
    # Landmark
    p_leye = draw_landmark_point(frame, lm, 2, CYAN)
    p_reye = draw_landmark_point(frame, lm, 5, CYAN)
    draw_line_between(frame, p_leye, p_reye, CYAN, 2)
    draw_landmark_point(frame, lm, 7, (180, 180, 0))
    draw_landmark_point(frame, lm, 8, (180, 180, 0))
    p_nose = draw_landmark_point(frame, lm, 0, WHITE, 5)
    p_lsho = draw_landmark_point(frame, lm, 11, GREEN)
    p_rsho = draw_landmark_point(frame, lm, 12, GREEN)
    draw_line_between(frame, p_lsho, p_rsho, GREEN, 2)
 
    mid_sho = ((p_lsho[0] + p_rsho[0]) // 2, (p_lsho[1] + p_rsho[1]) // 2)
    cv2.circle(frame, mid_sho, 5, WHITE, -1)
    draw_line_between(frame, p_nose, mid_sho, ORANGE, 2)
 
    mid_eye = ((p_leye[0] + p_reye[0]) // 2, (p_leye[1] + p_reye[1]) // 2)
    draw_line_between(frame, mid_eye, mid_sho, YELLOW, 2)
 
    # Box correzione
    if label != "TUP":
        bx, by = w // 2, h // 2
        ov2 = frame.copy()
        cv2.rectangle(ov2, (bx - 160, by - 35), (bx + 160, by + 35), (10, 10, 10), -1)
        cv2.addWeighted(ov2, 0.8, frame, 0.2, 0, frame)
        cv2.rectangle(frame, (bx - 160, by - 35), (bx + 160, by + 35), color, 2)
        put_text_pro(frame, f"CORREGGI: {desc}",
                     (bx - 148, by + 10), scale=0.6, color=color, thickness=1)
 
    return frame

## Calibration

In [ ]:
import collections, time, json, os
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# ════════════════════════════════════════════════════════════════════
# PARAMETERS
# ════════════════════════════════════════════════════════════════════
CALIBRATION_FRAMES = 30
CLASSIFICATION_FPS = 4
ALERT_SECONDS      = 5
ALERT_BUFFER_SIZE  = int(ALERT_SECONDS * CLASSIFICATION_FPS)  # 20 samples
ALERT_BAD_RATIO    = 0.70
WINDOW_NAME        = "Posture Monitor"
TEST_DIR           = "Test"
os.makedirs(TEST_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════════════
# PHASE 1 — CALIBRATION
# ════════════════════════════════════════════════════════════════════
z_buffer.clear()
baseline_stats = None

print("=" * 60)
print("PHASE 1: CALIBRATION")
print("Sit in correct posture and hold the position.")
print("=" * 60)
for s in range(5, 0, -1):
    print(f"  Starting in {s}...")
    time.sleep(1)

calib_collected = []
raw = capture_frames_batch(CALIBRATION_FRAMES * 2)
for frame in raw:
    if len(calib_collected) >= CALIBRATION_FRAMES:
        break
    try:
        res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    except Exception:
        continue
    if not res.pose_landmarks or not check_visibility(res):
        continue
    lm_arr = smooth_z(landmarks_to_array(res))
    fd, _  = extract_features_from_array(lm_arr, res)
    mv     = feature_dict_to_model_vector(fd, BEST_FEATURE_COLUMNS)
    if np.all(~np.isfinite(mv)):
        continue
    calib_collected.append(fd)
    print(f"  Valid frame {len(calib_collected)}/{CALIBRATION_FRAMES}")

if len(calib_collected) < CALIBRATION_FRAMES // 2:
    raise RuntimeError("Calibration failed — too few valid frames.")

baseline_stats = {}
for name in FEATURE_NAMES:
    vals  = np.array([d.get(name, np.nan) for d in calib_collected], dtype=float)
    valid = vals[np.isfinite(vals)]
    if len(valid) == 0:
        baseline_stats[name] = {"median": np.nan, "mean": np.nan, "std": np.nan}
    else:
        baseline_stats[name] = {
            "median": float(np.median(valid)),
            "mean":   float(np.mean(valid)),
            "std":    float(np.std(valid)),
        }

print("\n--- Baseline (median ± std) ---")
for name in FEATURE_NAMES:
    m  = baseline_stats[name]["median"]
    s  = baseline_stats[name]["std"]
    ms = f"{m:.4f}" if np.isfinite(m) else "NaN"
    ss = f"{s:.4f}" if np.isfinite(s) else "NaN"
    print(f"  {name:<40} {ms}  ±{ss}")

print("\n--- Oblique angle diagnostics ---")
sz_med = baseline_stats["shoulder_z_asym"]["median"]
sz_std = baseline_stats["shoulder_z_asym"]["std"]
sz_floor = STD_FLOOR["shoulder_z_asym"]
sz_std_safe = max(sz_std, sz_floor) if np.isfinite(sz_std) else sz_floor
if np.isfinite(sz_med):
    print(f"  shoulder_z_asym baseline: {sz_med:.3f}  std={sz_std:.3f}  floor={sz_floor}")
    print(f"  → expected noise in neutral posture: ±{sz_std_safe:.3f} "
          f"(±{sz_std_safe/sz_floor:.1f}x floor)")
    margin_tll = 1.8 * sz_std_safe
    print(f"  → deviation needed for TLL: >{sz_med + margin_tll:.3f} "
          f"or <{sz_med - margin_tll:.3f}")
    if sz_std > 0.20:
        print("  ⚠  very high std — calibration was unstable. Re-run it.")
    else:
        print("  ✓  Sufficient margin.")
else:
    print("  ⚠  shoulder_z_asym = NaN — check shoulder visibility.")

# ── trunk_forward_backward_angle diagnostics ──────────────────────
valid_tfba = [d.get("trunk_forward_backward_angle", np.nan)
              for d in calib_collected
              if np.isfinite(d.get("trunk_forward_backward_angle", np.nan))]
print(f"\n  trunk_forward_backward_angle valid in calibration: "
      f"{len(valid_tfba)}/{len(calib_collected)}")
if len(valid_tfba) < len(calib_collected) // 2:
    print("  ⚠  Less than 50% of frames have hips visible — "
          "trunk_fba will often be NaN → sagittal threshold will rise to 2.4.")
else:
    print("  ✓  Hips sufficiently visible.")

# ── General baseline quality check ───────────────────────────────
print("\n--- Baseline quality check ---")
STD_WARN = {
    "head_neck_vertical_angle":     4.0,
    "head_y_ratio":                 0.05,
    "trunk_forward_backward_angle": 5.0,
    "head_lateral_ratio":           0.04,
    "shoulder_z_asym":              0.20,
}
quality_ok = True
for name, threshold in STD_WARN.items():
    s = baseline_stats[name]["std"]
    if np.isfinite(s) and s > threshold:
        print(f"  ⚠  {name}: std={s:.4f} > {threshold} — noisy baseline")
        quality_ok = False
if quality_ok:
    print("  Stable baseline — you can proceed.")

print("\n✓ Calibration completed.")
print("→ Now run the debug cell or go directly to the main loop.")

## Debug

In [ ]:
EXPECTED_RANGES = {
    "head_tilt":                    (-20,  20),
    "shoulder_slope":               (-10,  10),
    "head_lateral_ratio":           (-0.5,  0.2),
    "head_y_ratio":                 (1.0,   2.0),
    "head_neck_vertical_angle":     (0,     25),
    "head_shoulder_alignment":      (-20,   20),
    "upper_body_inclination":       (-20,    5),
    "head_trunk_diff":              (-30,   30),
    "trunk_forward_backward_angle": (-20,   20),
}

print("Capturing frame...")
ret, dbg_frame = cap.read()
if not ret:
    raise RuntimeError("cap.read() failed.")

dbg_results = pose.process(cv2.cvtColor(dbg_frame, cv2.COLOR_BGR2RGB))
if not dbg_results.pose_landmarks:
    raise RuntimeError("No landmarks detected.")

dbg_lm        = dbg_results.pose_landmarks.landmark
dbg_lm_array  = landmarks_to_array(dbg_results)
dbg_lm_smooth = smooth_z(dbg_lm_array)
dbg_feat, dbg_ctrl = extract_features_from_array(dbg_lm_smooth, dbg_results)

# Visibility
LANDMARK_MAP = {0:"nose", 2:"l_eye", 5:"r_eye", 7:"l_ear", 8:"r_ear",
                11:"l_shoulder", 12:"r_shoulder", 23:"l_hip", 24:"r_hip"}
print("\n" + "═"*60)
print("§1  VISIBILITY")
print("═"*60)
all_vis = {}
for idx, name in LANDMARK_MAP.items():
    v = dbg_lm[idx].visibility
    all_vis[name] = v
    print(f"  {name:<15} {v:.3f}  {'✓' if v >= 0.5 else '✗ ← LOW'}")
hips_ok = all_vis["l_hip"] >= 0.5 and all_vis["r_hip"] >= 0.5
print(f"\n  → Hips visible? {'YES' if hips_ok else 'NO → trunk_fba = NaN'}")

# Reliability
print("\n" + "═"*60)
print("§2  RELIABILITY FLAGS")
print("═"*60)
for k, v in dbg_ctrl.items():
    print(f"  {k:<30} {v}")
trunk_ok = dbg_ctrl.get("trunk_support_reliable", False)
if not trunk_ok:
    print("\n  → trunk_fba = NaN, model uses imputer (training median).")

# Feature vs range
print("\n" + "═"*60)
print("§3  FEATURE vs EXPECTED RANGE")
print("═"*60)
print(f"  {'feature':<40} {'value':>9}  {'range':>20}  ok?")
print("  " + "─"*78)
n_nan, n_out = 0, 0
nan_feat, out_feat = [], []
for name in FEATURE_NAMES:
    val    = dbg_feat.get(name, np.nan)
    lo, hi = EXPECTED_RANGES.get(name, (None, None))
    rstr   = f"[{lo}, {hi}]" if lo is not None else "[—]"
    if not np.isfinite(val):
        n_nan += 1; nan_feat.append(name)
        print(f"  {name:<40} {'NaN':>9}  {rstr:>20}  —")
        continue
    ok = (lo is None) or (lo <= val <= hi)
    if not ok:
        n_out += 1; out_feat.append((name, val, lo, hi))
    print(f"  {name:<40} {val:>9.4f}  {rstr:>20}  {'✓' if ok else '⚠'}")

# Delta from baseline
print("\n" + "═"*60)
print("§3b  DELTA FROM BASELINE  (how far from calibrated neutral posture)")
print("═"*60)
if 'baseline_stats' in dir() and baseline_stats:
    print(f"  {'feature':<40} {'val':>8}  {'baseline':>9}  {'delta/std':>9}  signal")
    print("  " + "─"*78)
    for name in FEATURE_NAMES:
        val = dbg_feat.get(name, np.nan)
        b   = baseline_stats[name]["median"]
        std = baseline_stats[name]["std"]
        if not np.isfinite(val) or not np.isfinite(b):
            print(f"  {name:<40} {'NaN':>8}  {'—':>9}  {'—':>9}")
            continue
        std_safe   = max(std, STD_FLOOR.get(name, 0.001))
        delta_std  = (val - b) / std_safe
        if abs(delta_std) > 2.0:
            flag = "  ⚠ STRONG"
        elif abs(delta_std) > 1.0:
            flag = "  · medium"
        else:
            flag = ""
        print(f"  {name:<40} {val:>8.3f}  {b:>9.3f}  {delta_std:>9.2f}σ{flag}")
    print()
    print("  Active thresholds in classify_posture():")
    print(f"    SAG_ABS_THR  = 1.4σ  →  trunk_fba + head_y_ratio + head_neck_vert")
    print(f"    LAT_ABS_THR  = 2.0σ  →  head_lateral_ratio + upper_body_inclination")
    print(f"    ML_TRUST_THR = 0.85  →  below this confidence geometric rules kick in")
else:
    print("  ⚠  baseline_stats not available.")
    print("     Run PHASE 1 (calibration) in Cell 65 first.")

# Summary
print("\n" + "═"*60)
print("§4  SUMMARY")
print("═"*60)
print(f"\n  NaN ({n_nan}/{len(FEATURE_NAMES)}):")
if nan_feat:
    for f in nan_feat:
        tag = " [support]" if f == "trunk_forward_backward_angle" else " [PRIMARY ← issue]"
        print(f"    · {f}{tag}")
else:
    print("    None ✓")
print(f"\n  Out of range ({n_out}):")
if out_feat:
    for fname, fval, lo, hi in out_feat:
        print(f"    · {fname} = {fval:.4f}  (expected [{lo}, {hi}])")
else:
    print("    All within range ✓")

# Alignment
print("\n" + "═"*60)
print("§5  FEATURE_NAMES / BEST_FEATURE_COLUMNS ALIGNMENT")
print("═"*60)
if set(FEATURE_NAMES) == set(BEST_FEATURE_COLUMNS):
    print(f"  ✓ Aligned — {len(FEATURE_NAMES)} features.")
else:
    print("  ⚠ NOT aligned — reload .pkl files.")

# Frame
dbg_disp = dbg_frame.copy()
mp_drawing.draw_landmarks(dbg_disp, dbg_results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
    landmark_drawing_spec=mp_drawing.DrawingSpec(color=(0,220,0), thickness=2, circle_radius=3),
    connection_drawing_spec=mp_drawing.DrawingSpec(color=(200,200,200), thickness=1))
plt.figure(figsize=(9, 6))
plt.imshow(cv2.cvtColor(dbg_disp, cv2.COLOR_BGR2RGB))
plt.title(f"Debug | trunk_ok={trunk_ok} | NaN={n_nan}/{len(FEATURE_NAMES)} | out of range={n_out}",
          fontsize=10)
plt.axis("off"); plt.tight_layout(); plt.show()
print("\nDebug completed.")

## Loop

In [ ]:
# ════════════════════════════════════════════════════════════════════
# PHASE 2 — CONTINUOUS LOOP
# ════════════════════════════════════════════════════════════════════
if baseline_stats is None:
    raise RuntimeError("Run Cell 84 (calibration) first.")

# Clear the Z buffer from calibration before starting real monitoring
z_buffer.clear()

posture_buffer   = collections.deque(maxlen=ALERT_BUFFER_SIZE)
session_counts   = {cls: 0 for cls in label_encoder.classes_}
alert_active     = False
alert_label      = None
alert_start_time = None
bad_posture_secs = 0.0
total_frames     = 0
session_start    = time.time()
last_clf_time    = 0.0
loop_log         = []

current_label = "TUP"
current_conf  = 1.0
current_rule  = {
    "confidence_tier": "high",
    "reason":          [],
    "lateral_score":   None,
    "sagittal_score":  None,
    "eff_lat_thr":     None,
    "eff_sag_thr":     None,
}
last_lm = None

cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, 1280, 720)

print("Monitoring active — press Q on the window to stop.")
print(f"{'t':>7}  {'change':<16}  {'sag':>7}  {'lat':>7}  {'thr_s':>6}  {'thr_l':>6}  rule")
print("─" * 75)

try:
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            break

        now = time.time()

        if now - last_clf_time >= 1.0 / CLASSIFICATION_FPS:
            last_clf_time = now
            try:
                res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            except Exception:
                res = None

            if res and res.pose_landmarks and check_visibility(res):
                last_lm = res.pose_landmarks.landmark
                lm_arr  = smooth_z(landmarks_to_array(res))
                fd, _   = extract_features_from_array(lm_arr, res)
                mv      = feature_dict_to_model_vector(fd, BEST_FEATURE_COLUMNS)

                if not np.all(~np.isfinite(mv)):
                    lml, conf, _ = predict(mv, best_model, label_encoder)
                    lf, rule     = classify_posture(
                        lml, conf, fd, baseline_stats,
                        current_label=current_label
                    )

                    if lf != current_label:
                        sag    = rule.get("sagittal_score", "?")
                        lat    = rule.get("lateral_score",  "?")
                        thr_s  = rule.get("eff_sag_thr", "?")
                        thr_l  = rule.get("eff_lat_thr", "?")
                        reason = " | ".join(rule.get("reason", []))[:35]
                        t      = now - session_start
                        print(f"  {t:5.1f}s  "
                              f"{current_label} → {lf:<4}  "
                              f"sag={sag!s:>6}  lat={lat!s:>6}  "
                              f"ts={thr_s!s:>5}  tl={thr_l!s:>5}  {reason}")

                    current_label = lf
                    current_conf  = conf
                    current_rule  = rule

                    posture_buffer.append(lf)
                    session_counts[lf] = session_counts.get(lf, 0) + 1
                    total_frames += 1

                    loop_log.append({
                        "t":              round(now - session_start, 1),
                        "pred":           lf,
                        "ml":             lml,
                        "conf":           round(conf, 3),
                        "rule_triggered": rule.get("rule_triggered"),
                        "reason":         " | ".join(rule.get("reason", [])),
                        "lat_score":      rule.get("lateral_score"),
                        "sag_score":      rule.get("sagittal_score"),
                        "eff_lat_thr":    rule.get("eff_lat_thr"),
                        "eff_sag_thr":    rule.get("eff_sag_thr"),
                        **{
                            k: round(v, 4) if isinstance(v, float) and np.isfinite(v) else None
                            for k, v in fd.items()
                        },
                    })

        if len(posture_buffer) == ALERT_BUFFER_SIZE:
            bad       = [l for l in posture_buffer if l != "TUP"]
            bad_ratio = len(bad) / ALERT_BUFFER_SIZE

            if bad_ratio >= ALERT_BAD_RATIO:
                if not alert_active:
                    alert_active     = True
                    alert_start_time = now
                    from collections import Counter
                    alert_label = Counter(bad).most_common(1)[0][0]
                    print(f"⚠  ALERT: {alert_label} — "
                          f"{LABEL_DESC.get(alert_label, '')} "
                          f"({bad_ratio*100:.0f}% last 5s)")
            else:
                if alert_active:
                    bad_posture_secs += now - alert_start_time
                    print(f"✓  Correct posture — alert off "
                          f"(bad posture so far: {bad_posture_secs:.0f}s)")
                alert_active = False
                alert_label  = None

        # ── Display ──────────────────────────────────────────────
        display = frame.copy()
        h, w    = display.shape[:2]
        color   = LABEL_COLORS.get(current_label, (255, 255, 255))
        desc    = LABEL_DESC.get(current_label, current_label)

        # Semi-transparent top bar
        ov = display.copy()
        cv2.rectangle(ov, (0, 0), (w, 90), (10, 10, 10), -1)
        cv2.addWeighted(ov, 0.75, display, 0.25, 0, display)

        # Posture label
        cv2.putText(display, f"{current_label}  {desc}",
                    (14, 42), cv2.FONT_HERSHEY_DUPLEX, 1.0,
                    (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(display, f"{current_label}  {desc}",
                    (14, 42), cv2.FONT_HERSHEY_DUPLEX, 1.0,
                    color, 2, cv2.LINE_AA)

        # ML confidence
        cv2.putText(display, f"Conf: {current_conf:.1%}",
                    (14, 74), cv2.FONT_HERSHEY_DUPLEX, 0.55,
                    (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(display, f"Conf: {current_conf:.1%}",
                    (14, 74), cv2.FONT_HERSHEY_DUPLEX, 0.55,
                    (180, 180, 180), 1, cv2.LINE_AA)

        # Green/red side bar
        bar_col = (0, 220, 0) if current_label == "TUP" else (0, 0, 255)
        cv2.rectangle(display, (w - 18, 0), (w, h), bar_col, -1)

        # Main landmarks
        if last_lm:
            for idx, col in [(2,  (255, 220, 0)),
                             (5,  (255, 220, 0)),
                             (11, (0, 220, 0)),
                             (12, (0, 220, 0)),
                             (0,  (255, 255, 255))]:
                px = int(last_lm[idx].x * w)
                py = int(last_lm[idx].y * h)
                cv2.circle(display, (px, py), 6, col, -1)
                cv2.circle(display, (px, py), 8, (255, 255, 255), 1)

        # Bottom debug overlay: sag, lat, effective thresholds, reason
        if current_rule:
            sag      = current_rule.get("sagittal_score", "?")
            lat      = current_rule.get("lateral_score",  "?")
            thr_s    = current_rule.get("eff_sag_thr", "?")
            thr_l    = current_rule.get("eff_lat_thr", "?")
            rule_str = " | ".join(current_rule.get("reason", []))[:65]

            # Row 1: numeric scores + effective thresholds
            dbg1 = f"sag:{sag!s}(thr={thr_s!s})  lat:{lat!s}(thr={thr_l!s})"
            cv2.putText(display, dbg1,
                        (10, h - 22), cv2.FONT_HERSHEY_SIMPLEX, 0.38,
                        (0, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(display, dbg1,
                        (10, h - 22), cv2.FONT_HERSHEY_SIMPLEX, 0.38,
                        (180, 255, 180), 1, cv2.LINE_AA)

            # Row 2: classification reason
            cv2.putText(display, rule_str,
                        (10, h - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.38,
                        (0, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(display, rule_str,
                        (10, h - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.38,
                        (180, 255, 180), 1, cv2.LINE_AA)

        cv2.imshow(WINDOW_NAME, display)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Monitoring stopped.")
            cv2.destroyAllWindows()
            cv2.waitKey(1)
            break

finally:
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    if alert_active and alert_start_time:
        bad_posture_secs += time.time() - alert_start_time

# ════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ════════════════════════════════════════════════════════════════════
total_time = time.time() - session_start
good_time  = max(0.0, total_time - bad_posture_secs)

print("\n" + "═" * 60)
print("SESSION REPORT")
print("═" * 60)
print(f"  Total duration:      {total_time/60:.1f} min ({total_time:.0f}s)")
print(f"  Correct posture:     {good_time:.0f}s  ({good_time/total_time*100:.1f}%)")
print(f"  Bad posture:         {bad_posture_secs:.0f}s  "
      f"({bad_posture_secs/total_time*100:.1f}%)")
print(f"  Total samples:       {total_frames}")

if total_frames > 0:
    print("\n  Class distribution:")
    for cls in sorted(session_counts):
        cnt = session_counts[cls]
        if cnt > 0:
            pct = cnt / total_frames * 100
            bar = "█" * int(pct / 3)
            print(f"    {cls}  {LABEL_DESC.get(cls, cls):<30} {pct:5.1f}%  {bar}")

if total_frames > 0:
    labels_plot = [c for c in sorted(session_counts) if session_counts[c] > 0]
    values_plot = [session_counts[c] / total_frames * 100 for c in labels_plot]
    colors_plot = [
        (0/255, 220/255, 0/255)   if l == "TUP" else
        (255/255, 100/255, 0/255) if l in ("TLL", "TLR") else
        (0/255, 180/255, 255/255)
        for l in labels_plot
    ]
    plt.figure(figsize=(7, 4))
    plt.bar(labels_plot, values_plot, color=colors_plot)
    plt.ylabel("% of time")
    plt.title(f"Posture distribution — {total_time/60:.1f} min")
    plt.ylim(0, 100)
    for i, v in enumerate(values_plot):
        plt.text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(TEST_DIR, "session_report.png"), dpi=150)
    plt.show()
    print(f"\n✓ Chart saved to: {TEST_DIR}/session_report.png")

log_path = os.path.join(
    TEST_DIR, f"monitor_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)
with open(log_path, "w") as f:
    json.dump({
        "total_seconds":     round(total_time, 1),
        "good_seconds":      round(good_time, 1),
        "bad_seconds":       round(bad_posture_secs, 1),
        "total_frames":      total_frames,
        "class_counts":      session_counts,
        "alert_threshold_s": ALERT_SECONDS,
        "log":               loop_log,
    }, f, indent=2)
print(f"Log saved: {log_path}")

## Closing connection

In [ ]:
cap.release()
cv2.destroyAllWindows()
print("Connection closed.")